In [24]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence, compute_bpc

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
from collections import deque
import random 
import pickle 
import os
import zipfile

In [25]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # works only with NVIDIA GPUs (not on Mac)
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [76]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, context_size, context_projection_size=0, use_embedding=False, embedding_dim=10):
        super().__init__()
        self.use_embedding = use_embedding

        if context_projection_size != 0:
            self.context_layer = nn.Linear(context_size, context_projection_size)

        if self.use_embedding:
            self.embedding = nn.Embedding(vocab_size, embedding_dim)
            self.rnn = nn.RNN(embedding_dim+context_projection_size, hidden_size, batch_first=True, nonlinearity='tanh')
        else:
            self.input_projection = nn.Linear(vocab_size, embedding_dim)
            self.rnn = nn.RNN(embedding_dim+context_projection_size, hidden_size, batch_first=True, nonlinearity='tanh')
        
        self.linear = nn.Linear(hidden_size, vocab_size)

        ### use orthogonal initialization of weights ### 
        for name, param in self.rnn.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    
    def forward(self, x, context=None, h=None):
        if context != None:
            context_out = self.context_layer(context)
            if self.use_embedding:
                embedded = self.embedding(x)
                # print(embedded.size(), context_out.size())
                context_expanded = context_out.unsqueeze(1).expand(-1, embedded.size(1), -1)
                embedded = torch.cat((embedded, context_expanded), dim=2)
                out, h = self.rnn(embedded, h)
            else:
                embedded = self.input_projection(x)
                context_expanded = context_out.unsqueeze(1).expand(-1, embedded.size(1), -1)
                embedded = torch.cat((embedded, context_expanded), dim=2)
                out, h = self.rnn(embedded, h)
        else:
            if self.use_embedding:
                embedded = self.embedding(x)
                out, h = self.rnn(embedded, h)
            else:
                embedded = self.input_projection(x)
                out, h = self.rnn(embedded, h)

        out = self.linear(out[:,-1,:])

        return out, h  
    
class RNNDecoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, use_embedding=False, embedding_dim=10):
        super().__init__()
        self.use_embedding = use_embedding

        if self.use_embedding:
            self.embedding = nn.Embedding(vocab_size, embedding_dim)
            self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='tanh')
        else:
            self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True, nonlinearity='tanh')

        self.fc = nn.Linear(hidden_size, vocab_size)

        ### use orthogonal initialization of weights ###
        for name, param in self.rnn.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    
    def forward(self, x, h):
        if self.use_embedding:
            embedded = self.embedding(x)
            out, _ = self.rnn(embedded, h)
        else:
            out, _ = self.rnn(x, h)

        return self.fc(out) 
    
class RNNAutoencoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, context_size, context_projection_size, use_embedding=False, embedding_dim=10 ):
        super().__init__()
        self.encoder = RNNEncoder(vocab_size, hidden_size, context_size, context_projection_size, use_embedding, embedding_dim)
        self.decoder = RNNDecoder(vocab_size, hidden_size, use_embedding, embedding_dim)
    
    def forward(self, x, decoder_input, context, h=None):
        next_token, h = self.encoder(x, context, h)
        decoder_output = self.decoder(decoder_input, h)
        return next_token, decoder_output, h

In [77]:
def path_finder_loss(y_pred, decoder_output, y_target, decoder_target):
    ce1 = F.cross_entropy(y_pred.view(-1, y_pred.size(-1)), y_target.view(-1))
    ce2 = F.cross_entropy(decoder_output.view(-1, decoder_output.size(-1)), decoder_target.view(-1))

    return (ce1+ce2)/2

In [64]:
def path_finder_loss_mse(y_pred, decoder_output, y_target, decoder_target):
    ce1 = F.mse_loss(y_pred.view(-1), y_target.view(-1))
    ce2 = F.mse_loss(decoder_output.view(-1), decoder_target.view(-1))

    return (ce1+ce2)/2

In [ ]:
def train_layer(model, optimizer, criterion, X, y, context=None, h=None, layer=0):
    if layer == 0:
        decoder_input = torch.cat([torch.zeros((1, 1), dtype=torch.long), X[:, :-1]], dim=1)
    else:
        decoder_input = torch.cat([torch.zeros((1, 1, X.shape[2]), dtype=torch.long), X[:, :-1]], dim=1)

    decoder_target = X.flip(1)

    optimizer.zero_grad()
    y_pred, decoder_output, h = model(X, decoder_input, context, h)
    # print(y_pred[0], 'y_pred', y[0], 'y', decoder_output[0], decoder_target[0])
    loss = criterion(y_pred[0], decoder_output[0], y[0], decoder_target[0])     
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if layer == 0:
        with torch.no_grad():
            if y[0] == y_pred[0].argmax():
                correct = 1
            else:
                correct = 0
        
        return correct, loss.detach(), y_pred, h.detach()
    
    return h.detach()
        

In [57]:
# Step 1: Download and extract text8
def download_text8(path="dataset/text8.zip"):
    url = "http://mattmahoney.net/dc/text8.zip"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        print("Downloading text8...")
        urllib.request.urlretrieve(url, path)
    with zipfile.ZipFile(path) as zf:
        data = zf.read(zf.namelist()[0]).decode('utf-8')
    return data

# Step 2: Build character-level vocabulary
def build_vocab(text):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos

# Step 3: Encode text into integer tokens
def encode(text, stoi):
    return np.array([stoi[c] for c in text], dtype=np.int32)

# Step 4: Create input and target sequences
#def create_dataset(encoded_text, seq_len=64, step=1):
#    inputs, targets = [], []
#    for i in range(0, len(encoded_text) - seq_len, step):
#        inputs.append(encoded_text[i:i+seq_len])
#        targets.append(encoded_text[i+1:i+seq_len+1])
#    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

class Dataset_converter(Dataset):
    def __init__(self, encoded_text, working_memory=1, short_term_memory=8):
        
        self.X = []
        self.y = []

        for ii in range(0, len(encoded_text) - working_memory - short_term_memory, 1):
            self.X.append(
                encoded_text[ii:ii+short_term_memory]
            )
            self.y.append(
                encoded_text[ii+short_term_memory]
            )

        self.X = tnsr(np.array(self.X)).long()
        self.y = tnsr(np.array(self.y)).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [80]:
text = download_text8()
stoi, itos = build_vocab(text)
encoded = encode(text, stoi)
train_sample = 90000000
test_sample = 10000000
short_term_memory = 4

train_data_set = Dataset_converter(encoded[:train_sample], 1, short_term_memory=short_term_memory)
test_data_set = Dataset_converter(encoded[train_sample:], 1, short_term_memory=short_term_memory)

In [94]:
reps = 1
res = []
res_bpc = []

for rep in range(reps):
    print('Doing rep ', rep)

    total_layers = 6
    model = {}
    input_buffer = {}
    target_buffer = {}
    context = {}
    h = {}
    vocab_size = 27
    embedding_dim = 50
    context_projection_size = [50, 50, 50, 50, 50]
    downsample_factor = short_term_memory
    lr = [1e-4,1e-4,1e-4,1e-4,1e-4, 1e-4]
    hidden_size = [256, 256, 256, 256, 256, 256]
    optimizers = []
    criteria = []
    lr_scheduler = []

    for layer in range(total_layers):  
        if layer == 0:
            model[layer] = RNNAutoencoder(vocab_size, hidden_size[layer], hidden_size[layer+1], context_projection_size[layer], use_embedding=True, embedding_dim=embedding_dim)
            criteria.append(
                path_finder_loss
            )
        else:
            if layer == total_layers-1:
                model[layer] = RNNAutoencoder(hidden_size[layer-1], hidden_size[layer], 0, context_projection_size=0, embedding_dim=embedding_dim)
            else:
                model[layer] = RNNAutoencoder(hidden_size[layer-1], hidden_size[layer], hidden_size[layer+1], context_projection_size[layer], embedding_dim=embedding_dim)

            context[layer-1] = torch.zeros((1,hidden_size[layer]))
            input_buffer[layer] = deque(
                [torch.zeros((hidden_size[layer-1])) for _ in range(short_term_memory)],
                maxlen=short_term_memory
            )
            target_buffer[layer] = torch.zeros((hidden_size[layer-1]))
            criteria.append(
                path_finder_loss_mse
            )

        optimizers.append(
            torch.optim.Adam(model[layer].parameters(), lr=lr[layer], weight_decay=1e-6)
        )
        lr_scheduler.append(
            torch.optim.lr_scheduler.CosineAnnealingLR(optimizers[-1], T_max=10000)
        )

        h[layer] = None

    context[layer] = None



    ### handle data ###
    train_loader = DataLoader(train_data_set, batch_size=1, shuffle=False)

    ### train the model ###
    total_avg = 10000
    correct = np.zeros(total_avg,dtype=float)
    bpc = np.zeros(total_avg,dtype=float)
    # decoder_correct = np.zeros(1000,dtype=float)
    test_acc = []
    test_bpc = []

    total = 1
    epochs = 1
    ls = 0
    for epoch in range(epochs):
        for X, y in train_loader:
            for layer in range(total_layers):
                # print(layer)

                if layer == 0:
                    accurate, l, logits, h[layer] = train_layer(model[0], optimizers[0], criteria[0], X, y, context[0], h[layer])

                    correct[total%total_avg] = accurate
                    bpc[total%total_avg] = compute_bpc(logits, y)
                    ls += l

                elif total%(downsample_factor**layer)==0:
                    input_buffer[layer].append(
                        target_buffer[layer].clone()
                    )
                    target_buffer[layer] = h[layer-1][0][0]
                    
                    h[layer] = train_layer(
                        model[layer], optimizers[layer], criteria[layer], torch.stack(list(input_buffer[layer])).unsqueeze(0), 
                        torch.stack(list(target_buffer[layer])).unsqueeze(0), context[layer], h[layer], layer=layer
                    )
                    context[layer-1][0] = h[layer][0][0]

                    
                if total%total_avg==0:
                    lr_scheduler[layer].step()

            total += 1
            
            test_acc.append(
                    np.sum(correct)/total if total<total_avg else np.sum(correct)/total_avg
                )
            test_bpc.append(
                np.sum(bpc)/total if total<total_avg else np.sum(bpc)/total_avg
            )
            if total%total_avg == 0:
                print(f'Iter : {total+1}, accuracy: {test_acc[-1]:.4f}, bpc: {test_bpc[-1]:.4f}, loss:  {ls/total_avg:.4f}')
                ls = 0
                

    # res.append(
    #     test_acc
    # )
    # res_bpc.append(
    #     test_bpc
    # )


Doing rep  0
Iter : 10001, accuracy: 0.3061, bpc: 3.4756, loss:  1.4506
Iter : 20001, accuracy: 0.3930, bpc: 3.1031, loss:  1.0825
Iter : 30001, accuracy: 0.4147, bpc: 2.9764, loss:  1.0342
Iter : 40001, accuracy: 0.4209, bpc: 2.9701, loss:  1.0310
Iter : 50001, accuracy: 0.4230, bpc: 2.9551, loss:  1.0253
Iter : 60001, accuracy: 0.4357, bpc: 2.8929, loss:  1.0031
Iter : 70001, accuracy: 0.4641, bpc: 2.7828, loss:  0.9663
Iter : 80001, accuracy: 0.4770, bpc: 2.7163, loss:  0.9422
Iter : 90001, accuracy: 0.4775, bpc: 2.7899, loss:  0.9678
Iter : 100001, accuracy: 0.4596, bpc: 2.8621, loss:  0.9933
Iter : 110001, accuracy: 0.4650, bpc: 2.8107, loss:  0.9752
Iter : 120001, accuracy: 0.4494, bpc: 2.9010, loss:  1.0067
Iter : 130001, accuracy: 0.4592, bpc: 2.8353, loss:  0.9835
Iter : 140001, accuracy: 0.4566, bpc: 2.8380, loss:  0.9843
Iter : 150001, accuracy: 0.4855, bpc: 2.7081, loss:  0.9395
Iter : 160001, accuracy: 0.5317, bpc: 2.4591, loss:  0.8527
Iter : 170001, accuracy: 0.5019, bpc

In [95]:
with open('model_text8.pickle', 'wb') as f:
    pickle.dump(model, f)

In [111]:
for layer in range(total_layers):  
    if layer != 0:
        context[layer-1] = torch.zeros((1,hidden_size[layer]))
        input_buffer[layer] = deque(
            [torch.zeros((hidden_size[layer-1])) for _ in range(short_term_memory)],
            maxlen=short_term_memory
        )
        target_buffer[layer] = torch.zeros((hidden_size[layer-1]))

    h[layer] = None

context[layer] = None


total = 0
bpc_test = 0
test_loader = DataLoader(test_data_set, batch_size=1, shuffle=False)

for X, y in tqdm(test_loader):
    with torch.no_grad():
        for layer in range(total_layers):
            # print(layer)

            if layer == 0:
                logits, h[layer] = model[0].encoder(X, context[0], h[layer]) 

                bpc_test += compute_bpc(logits, y)

            elif total%(downsample_factor**layer)==0:
                input_buffer[layer].append(
                    target_buffer[layer].clone()
                )
                target_buffer[layer] = h[layer-1][0][0]
                
                _, h[layer] = model[layer].encoder(torch.stack(list(input_buffer[layer])).unsqueeze(0), context[layer], h[layer])
                context[layer-1][0] = h[layer][0][0]

                
            # if total%total_avg==0:
            #     lr_scheduler[layer].step()

        total += 1
        
print(f'Finall BPC on test set: {bpc_test/total:.4f}')

100%|██████████| 9999995/9999995 [29:12<00:00, 5705.02it/s]

Finall BPC on test set: 2.4261
